# 09.6 - Inference & Decoding

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Inference is using a trained model to generate text. **Decoding strategies** choose how the model selects tokens from the probability distribution at each step: greedy (most likely token), beam search (explore multiple candidates), and sampling (weighted randomness). The same model can produce very different outputs depending on strategy.

## 2. Why Does This Matter?

Matching decoding to your task controls quality, diversity, and coherence. Greedy is good for code/facts; sampling is good for chat and creative writing; beam search is good for translation. Choosing wrong settings produces repetitive, incoherent, or dull output.

## 3. Prerequisites

- Unit 09.1 (language models)
- Unit 09.5 (pretraining)

## 4. Learning Objectives

- Implement greedy and beam search from scratch
- Implement top-k and top-p sampling
- Apply temperature scaling
- Choose a decoding strategy for a task

## 5. Mental Model

Decoding is navigating a branching tree of possible sentences. At each step the model offers a menu of next tokens with probabilities. The strategy is how you pick: always the top item (greedy), explore several paths (beam), or roll a weighted die (sampling).

```text
logits -> softmax -> distribution over vocab -> strategy picks a token -> append -> repeat
```


## 6. Setup

We use synthetic logits so the decoding algorithms are easy to inspect. No model needed.


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import torch.nn.functional as F
torch.manual_seed(7)

# A fixed 'next-token' distribution over a 10-token vocabulary (a stand-in for model logits)
logits = torch.tensor([1.2, 3.0, 0.4, 2.5, 0.1, 4.0, 1.8, 0.9, 2.2, 3.5])
probs = F.softmax(logits, dim=-1)
print("Token probabilities:")
for i, p in enumerate(probs):
    print(f"  token {i}: {p.item():.4f}")


Token probabilities:
  token 0: 0.0231
  token 1: 0.1400
  token 2: 0.0104
  token 3: 0.0849
  token 4: 0.0077
  token 5: 0.3807
  token 6: 0.0422
  token 7: 0.0171
  token 8: 0.0629
  token 9: 0.2309


## 7. Greedy Decoding

Always take the token with the highest probability. Deterministic, but can be repetitive over many steps.


In [2]:
def greedy(probs):
    return int(probs.argmax().item())

tok = greedy(probs)
print(f"Greedy picks token {tok} with probability {probs[tok].item():.3f}")
print("Always the max -> deterministic.")


Greedy picks token 5 with probability 0.381
Always the max -> deterministic.


## 8. Sampling

Sample from the distribution proportional to probability. Stochastic - different draws each time.


In [3]:
def sample(probs, n=10):
    return [int(torch.multinomial(probs, 1).item()) for _ in range(n)]

draws = sample(probs)
print("10 samples:", draws)
print("High-prob tokens appear more often, but low-prob tokens sometimes win.")


10 samples: [1, 6, 8, 5, 5, 0, 9, 9, 4, 9]
High-prob tokens appear more often, but low-prob tokens sometimes win.


## 9. Temperature Scaling

Divide logits by temperature before softmax: T<1 sharpens (more deterministic), T>1 flattens (more random).


In [4]:
def apply_temp(logits, T):
    return F.softmax(logits / T, dim=-1)

for T in [0.3, 1.0, 3.0]:
    p = apply_temp(logits, T)
    top = int(p.argmax())
    print(f"T={T}: top={top} p={p[top]:.3f}  entropy={-(p*torch.log(p+1e-9)).sum():.3f}")
print("Higher T -> higher entropy (more uniform / random).")


T=0.3: top=5 p=0.810  entropy=0.606
T=1.0: top=5 p=0.381  entropy=1.740
T=3.0: top=5 p=0.181  entropy=2.218
Higher T -> higher entropy (more uniform / random).


## 10. Top-k Sampling

Keep only the k most probable tokens, renormalize, sample from those.


In [5]:
def top_k(probs, k):
    topk_vals, topk_idx = torch.topk(probs, k)
    # zero out everything not in top-k, then renormalize
    filtered = torch.zeros_like(probs)
    filtered[topk_idx] = topk_vals
    return filtered / filtered.sum()

p3 = top_k(probs, 3)
print("Top-3 filtered probabilities (renormalized):")
for i, p in enumerate(p3):
    if p > 0:
        print(f"  token {i}: {p.item():.4f}")
print("Only the top 3 tokens can ever be sampled.")


Top-3 filtered probabilities (renormalized):
  token 1: 0.1863
  token 5: 0.5065
  token 9: 0.3072
Only the top 3 tokens can ever be sampled.


## 11. Top-p (Nucleus) Sampling

Sort by probability and keep the smallest set whose cumulative probability >= p, then renormalize.


In [6]:
def top_p(probs, p):
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    keep = (cumsum - sorted_probs) < p   # include token only if accumulated BEFORE it < p
    keep_mask = torch.zeros_like(probs)
    keep_mask[sorted_idx[keep]] = 1.0
    filtered = probs * keep_mask
    return filtered / filtered.sum()

p95 = top_p(probs, 0.9)
nz = int((p95 > 0).sum())
print(f"Top-p (0.9) keeps {nz} tokens")
for i, p in enumerate(p95):
    if p > 0:
        print(f"  token {i}: {p.item():.4f}")
print("Adaptive: the count depends on the distribution shape.")


Top-p (0.9) keeps 6 tokens
  token 1: 0.1487
  token 3: 0.0902
  token 5: 0.4043
  token 6: 0.0448
  token 8: 0.0668
  token 9: 0.2452
Adaptive: the count depends on the distribution shape.


## 12. Beam Search

Maintain `beam_width` best sequences. At each step expand every candidate, keep the top `beam_width` overall by cumulative log-probability. This explores multiple paths deterministically.


In [7]:
import numpy as np

def beam_search(step_fn, vocab_size, beam_width=3, steps=4, start=None):
    # each hypothesis: (sequence, cumulative log-prob)
    hypotheses = [([] if start is None else list(start), 0.0)]
    for step in range(steps):
        all_next = []
        for seq, score in hypotheses:
            logits = step_fn(seq)               # model logits given current seq
            logp = F.log_softmax(logits, dim=-1)
            topk_vals, topk_idx = torch.topk(logp, beam_width)
            for v, i in zip(topk_vals, topk_idx):
                all_next.append((seq + [int(i)], score + float(v)))
        hypotheses = sorted(all_next, key=lambda x: x[1], reverse=True)[:beam_width]
    best_seq = max(hypotheses, key=lambda x: x[1])[0]
    return best_seq, hypotheses

# step_fn returns random logits (stands in for a model) - identical for any seq
def step_fn(seq):
    torch.manual_seed(3)
    return torch.randn(vocab_size)

vocab_size = 6
best, hyps = beam_search(step_fn, vocab_size, beam_width=3, steps=3)
print("Beam width 3, 3 steps.")
print("Top-3 final candidates and their cumulative log-probs:")
for seq, score in hyps:
    print(f"  {seq}  logp={score:.3f}")
print("Best:", best)


Beam width 3, 3 steps.
Top-3 final candidates and their cumulative log-probs:
  [0, 0, 0]  logp=-3.140
  [0, 0, 1]  logp=-3.769
  [0, 1, 0]  logp=-3.769
Best: [0, 0, 0]


## 13. Comparing Strategies on a Tiny Trained Model

Reuse the tiny pretrained model from 09.5 to generate with different strategies. (Defined inline here so this notebook is self-contained.)


In [8]:
def decode_sequence(logits_fn, start, strategy='greedy', max_new=6, **kw):
    ids = list(start)
    for _ in range(max_new):
        lg = logits_fn(ids)
        lp = F.softmax(lg, dim=-1)
        if strategy == 'greedy':
            tok = int(lp.argmax())
        elif strategy == 'topk':
            tok = int(torch.multinomial(top_k(lp, kw.get('k', 3)), 1))
        elif strategy == 'topp':
            tok = int(torch.multinomial(top_p(lp, kw.get('p', 0.9)), 1))
        else:
            tok = int(torch.multinomial(lp, 1))
        ids.append(tok)
    return ids

# Pure synthetic distribution from token 0
def make_logits(ids):
    torch.manual_seed(11)
    return torch.randn(8) + 1.0

start = [0]
print("Greedy:", decode_sequence(make_logits, start, 'greedy'))
print("Top-k  :", decode_sequence(make_logits, start, 'topk', k=3))
print("Top-p  :", decode_sequence(make_logits, start, 'topp', p=0.9))
print("Sample:", decode_sequence(make_logits, start, 'sample'))
print("\nNote the token-level variety across strategies.")


Greedy: [0, 1, 1, 1, 1, 1, 1]


Top-k  : [0, 7, 7, 7, 7, 7, 7]
Top-p  : [0, 7, 7, 7, 7, 7, 7]


Sample: [0, 7, 7, 7, 7, 7, 7]

Note the token-level variety across strategies.


## 14. Failure Case & Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Repetitive output | greedy / no rep penalty | sampling or repetition penalty |
| Incoherent output | temperature too high | reduce temperature 0.7-1.0 |
| Output too short | stop token early | adjust max length / stop |
| Output too long | no length limit | set max_new_tokens |

## 15. Real-World Considerations

- Chat apps: sampling (top-p + temperature) to avoid boring repetition.
- Code tools: greedy or low temperature for correctness.
- Translation/summarization: beam search for coherence.
- Always set a max length; log decoding params for reproducibility.

## 16. Common Mistakes

- Greedy for creative tasks (repetitive).
- High temperature for factual tasks (unreliable).
- No max length (runaway generation).
- No repetition penalty when sampling.

## 17. When NOT to Use Certain Strategies

- Avoid greedy when you need diversity.
- Avoid sampling when you need deterministic, verifiable output.
- Avoid beam search for real-time chat (slow, low diversity).

## 18. Challenge

Implement a repetition penalty: subtract a penalty from logits of tokens already generated, then sample. Confirm repeated tokens become less likely.


In [9]:
def penalized_sampling(step_fn, start, penalty=1.5, max_new=6):
    ids = list(start)
    for _ in range(max_new):
        lg = step_fn(ids).clone()
        # frequency penalty: push down logits of already-generated tokens
        for t in ids:
            lg[t] = lg[t] - penalty
        p = F.softmax(lg, dim=-1)
        ids.append(int(torch.multinomial(p, 1)))
    return ids

# A step_fn that always prefers token 2 (adversarially repetitive)
def repetitive(seq):
    lg = torch.zeros(6)
    lg[2] = 5.0   # token 2 strongly preferred
    return lg

print("Without penalty:", penalized_sampling(repetitive, [0], penalty=0.0))
print("With penalty   :", penalized_sampling(repetitive, [0], penalty=3.0))
print("Penalty makes already-used tokens less likely -> breaks repetition.")


Without penalty: [0, 2, 2, 2, 2, 2, 2]
With penalty   : [0, 2, 5, 2, 2, 4, 3]
Penalty makes already-used tokens less likely -> breaks repetition.


## 19. Closed-Book Recall

1. What is the difference between greedy and beam search?
2. How does temperature affect the output distribution?
3. When should you use sampling vs deterministic decoding?
4. What is the role of stop tokens?

## 20. Teach-Back Questions

Explain to another person:

- Beam search in 3-4 sentences.
- Why greedy output tends to be repetitive.

## 21. Summary

You implemented greedy, sampling, temperature, top-k, top-p, and beam search from scratch, then compared them and added a repetition penalty.

## 22. Further Experiment

- Use beam search on a real trained model.
- Tune the repetition penalty strength and observe repetition vs variety.

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
